In [61]:
# 单元格 2：导入库
# 说明：requests 负责请求智联前端 JSON 接口，pandas 负责表格处理，tqdm 负责显示进度条。

import json
import random
import re
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urlencode

import pandas as pd
import requests
from tqdm.auto import tqdm


## 单元格 3：自定义搜索参数

最常改的地方都在这里：关键词、城市列表、页数、暂停时间。比如要抓“后端开发”，把 `KEYWORD` 改成 `后端开发`；如果只想抓重庆，把 `CITY_LIST` 改成 `['重庆']`。


In [62]:
# 单元格 3：自定义搜索参数
# 你主要改这里：KEYWORD、CITY_LIST、START_PAGE、MAX_PAGES。

KEYWORD = '数据开发'        # 搜索关键词，例如：'Python'、'数据分析'、'运维'

# 按网页上方城市入口批量抓取；每个城市都会抓 START_PAGE 到 START_PAGE + MAX_PAGES - 1。
CITY_LIST = [
    '全国'
]
# CITY_LIST = [
#     '北京', '上海', '广州', '深圳', '天津', '武汉', '西安', '成都',
#     '大连', '长春', '沈阳', '南京', '济南', '青岛', '杭州', '苏州',
#     '无锡', '宁波', '重庆', '郑州', '长沙', '福州', '厦门', '哈尔滨'
# ]

# 兼容单城市调试：如果只想抓一个城市，把 CITY_LIST 改成 ['重庆'] 即可。
CITY = CITY_LIST[0]
CITY_CODE = None          # 单城市手动代码；批量城市通常保持 None，自动从 CITY_CODE_MAP 取

START_PAGE = 1            # 每个城市的起始页
MAX_PAGES = 50             # 每个城市最多抓取多少页；当前需求是每个城市 1-5 页
PAGE_SIZE = 20            # 每页条数，智联当前接口常用 20
EMPTY_PAGE_STOP = 2       # 连续多少个空页后停止，避免某城市没有足够页数时一直请求
PAGE_SLEEP_RANGE = (1.5, 3.0)  # 每页之间随机暂停，降低触发安全验证概率
REQUEST_TIMEOUT = 25      # 单次请求超时时间（秒）
REQUEST_RETRIES = 4       # 单页请求失败后的最大重试次数
RETRY_BACKOFF_BASE = 2.0  # 重试等待的基础秒数，会逐次递增
RETRY_SLEEP_RANGE = (0.8, 2.0)  # 每次重试额外增加的随机等待秒数

def locate_project_root():
    """定位当前项目根目录，避免从 notebook 子目录运行时把数据写到错误位置。"""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / '爬虫').is_dir() and (base / '数据清洗').is_dir():
            return base
        if base.name == '爬虫' and (base.parent / '数据清洗').is_dir():
            return base.parent
    for base in [cwd, *cwd.parents]:
        if (base / 'data').is_dir():
            return base
    raise FileNotFoundError('没有找到项目根目录，请在项目目录或 notebook 所在项目子目录下运行。')


PROJECT_ROOT = locate_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'data' # 输出到项目根目录下的 data，会自动创建

# 常用城市代码。若目标城市不在这里，在这里补充。
CITY_CODE_MAP = {
    '全国': '489',
    '北京': '530', '上海': '538', '天津': '531', '重庆': '551',
    '广州': '763', '深圳': '765', '武汉': '736', '西安': '854',
    '成都': '801', '大连': '600', '长春': '613', '沈阳': '599',
    '南京': '635', '济南': '702', '青岛': '703', '杭州': '653',
    '苏州': '639', '无锡': '636', '宁波': '654', '郑州': '719',
    '长沙': '749', '福州': '681', '厦门': '682', '哈尔滨': '622'
}

# 额外筛选条件。会直接合并进 JSON 请求体。
# order=4 是当前接口常见的排序值；如果后续失效，可删掉或改成接口支持的其他值。
EXTRA_PARAMS = {
    'order': 4,
}


## 单元格 4：请求头与安全验证检测

智联搜索页 HTML 当前容易返回安全验证页。本 notebook 改为请求前端 JSON 接口，并保留安全验证检测，避免把验证页误当成岗位数据。


In [63]:
# 单元格 4：请求头与安全验证检测
# 说明：搜索页 HTML 现在经常返回安全验证；这里改为请求智联前端 JSON 接口。

SEARCH_API_URL = 'https://fe-api.zhaopin.com/c/i/search/positions'

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/125.0.0.0 Safari/537.36'
    ),
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
    'Content-Type': 'application/json;charset=UTF-8',
    'Origin': 'https://www.zhaopin.com',
    'Referer': 'https://www.zhaopin.com/',
    'Connection': 'close',
    'x-zp-page-code': '4019',
    'x-zp-platform': '13',
    'x-zp-business-system': '1',
}

SECURITY_MARKERS = (
    'Security Verification',
    'TencentEOCaptcha',
    'EO-Bot-Captcha-Token',
    '正在验证连接安全性',
    'Protected by Tencent Cloud EdgeOne',
)


def make_request_session() -> requests.Session:
    """创建新的请求会话；断连后会重建，避免复用已失效连接。"""
    new_session = requests.Session()
    new_session.headers.update(HEADERS)
    return new_session


session = make_request_session()


def reset_request_session() -> None:
    """关闭旧会话并重建，处理 RemoteDisconnected 这类连接池断连。"""
    global session
    try:
        session.close()
    except Exception:
        pass
    session = make_request_session()


def looks_like_security_verification(text: str) -> bool:
    """判断返回内容是否像安全验证页面。"""
    return any(marker in text for marker in SECURITY_MARKERS)


## 单元格 5：城市代码、展示 URL 与接口请求体构造

`build_search_page_url` 用于记录和 Referer；真正取数的是 `build_search_payload` 构造出的 JSON 请求体。


In [64]:
# 单元格 5：城市代码、展示 URL 与接口请求体构造
# 说明：build_search_page_url 只用于记录和 Referer；真正取数用 build_search_payload。


def resolve_city_code(city: str, city_code: Optional[str] = None) -> str:
    """优先使用手动 CITY_CODE；否则从 CITY_CODE_MAP 根据城市名查代码。"""
    if city_code not in (None, ''):
        return str(city_code)
    if city in CITY_CODE_MAP:
        return CITY_CODE_MAP[city]
    raise ValueError(f'城市 {city!r} 不在 CITY_CODE_MAP 中，请手动设置 CITY_CODE。')


def build_search_page_url(
    keyword: str,
    city_code: str,
    page: int,
    extra_params: Optional[Dict[str, Any]] = None,
) -> str:
    """构造可读的搜索页 URL，用于记录和 Referer。"""
    params = {'kw': keyword, 'p': str(page)}
    if city_code:
        params['jl'] = str(city_code)
    if extra_params:
        for key, value in extra_params.items():
            if value not in (None, '') and key not in {'S_SOU_FULL_INDEX', 'S_SOU_WORK_CITY', 'pageIndex', 'pageSize'}:
                params[key] = value
    return 'https://www.zhaopin.com/sou/?' + urlencode(params)


def build_api_query_params() -> Dict[str, str]:
    """构造接口 URL 上的动态参数。"""
    return {
        '_v': f'{random.random():.8f}',
        'x-zp-page-request-id': f'{int(time.time() * 1000)}-{random.randint(100000, 999999)}',
        'x-zp-client-id': str(uuid.uuid4()),
    }


def build_search_payload(
    keyword: str,
    city_code: str,
    page: int,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """构造智联搜索 JSON 请求体。"""
    payload: Dict[str, Any] = {
        'S_SOU_FULL_INDEX': keyword,
        'pageIndex': int(page),
        'pageSize': int(page_size),
        'anonymous': 1,
        'eventScenario': 'pcSearchedSouSearch',
        'platform': 13,
        'version': '0.0.0',
        'order': 4,
    }
    if city_code:
        payload['S_SOU_WORK_CITY'] = str(city_code)
    if extra_params:
        payload.update({k: v for k, v in extra_params.items() if v not in (None, '')})
    return payload


# 先打印一个示例 URL 和请求体，确认关键词和城市参数是否正确。
resolved_city_code = resolve_city_code(CITY, CITY_CODE)
example_url = build_search_page_url(KEYWORD, resolved_city_code, START_PAGE, EXTRA_PARAMS)
example_payload = build_search_payload(KEYWORD, resolved_city_code, START_PAGE, PAGE_SIZE, EXTRA_PARAMS)
print(example_url)
print(json.dumps(example_payload, ensure_ascii=False, indent=2))


https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=489&order=4
{
  "S_SOU_FULL_INDEX": "数据开发",
  "pageIndex": 1,
  "pageSize": 20,
  "anonymous": 1,
  "eventScenario": "pcSearchedSouSearch",
  "platform": 13,
  "version": "0.0.0",
  "order": 4,
  "S_SOU_WORK_CITY": "489"
}


## 单元格 6：JSON 接口请求函数

这一格请求 `https://fe-api.zhaopin.com/c/i/search/positions`，并检查是否返回安全验证或异常 JSON。


In [65]:
# 单元格 6：JSON 接口请求函数
# 说明：搜索页 HTML 会被安全验证拦截；这里直接请求前端 JSON 接口并返回 data。

RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
RETRYABLE_REQUEST_EXCEPTIONS = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ChunkedEncodingError,
)


class PageRequestError(RuntimeError):
    """单页接口请求多次重试后仍失败。"""


def retry_wait_seconds(attempt: int) -> float:
    """计算逐次递增的重试等待时间。"""
    base = RETRY_BACKOFF_BASE * (2 ** max(attempt - 1, 0))
    jitter = random.uniform(*RETRY_SLEEP_RANGE)
    return min(base + jitter, 30.0)


def fetch_position_page(
    keyword: str,
    city_code: str,
    page: int,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
    timeout: int = 25,
    retries: Optional[int] = None,
) -> Tuple[Dict[str, Any], Dict[str, Any], str]:
    """请求一页职位 JSON，返回 (data, payload, api_url)。"""
    payload = build_search_payload(keyword, city_code, page, page_size, extra_params)
    page_url = build_search_page_url(keyword, city_code, page, extra_params)
    headers = dict(HEADERS)
    headers['Referer'] = page_url
    max_retries = REQUEST_RETRIES if retries is None else int(retries)
    last_error: Optional[BaseException] = None

    for attempt in range(1, max_retries + 1):
        api_params = build_api_query_params()
        try:
            resp = session.post(
                SEARCH_API_URL,
                params=api_params,
                headers=headers,
                data=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
                timeout=timeout,
            )

            if resp.status_code in RETRYABLE_STATUS_CODES:
                raise requests.exceptions.HTTPError(
                    f'HTTP {resp.status_code}: {resp.text[:200]!r}',
                    response=resp,
                )

            resp.raise_for_status()
            text = resp.text
            if looks_like_security_verification(text):
                raise RuntimeError('接口返回了网站安全验证页。请降低频率，稍后重试，或在浏览器中完成验证后再运行。')

            try:
                result = resp.json()
            except ValueError as exc:
                raise RuntimeError(f'接口没有返回 JSON，前 300 字符：{text[:300]!r}') from exc

            if result.get('code') != 200 or result.get('apiCode') not in (None, 200):
                raise RuntimeError(f'接口返回异常：{json.dumps(result, ensure_ascii=False)[:500]}')

            data = result.get('data') or {}
            if data.get('isVerification'):
                raise RuntimeError('接口提示需要安全验证，未继续抓取。')
            if not isinstance(data.get('list', []), list):
                raise RuntimeError(f'接口 data.list 不是列表：{json.dumps(data, ensure_ascii=False)[:500]}')

            return data, payload, resp.url

        except RETRYABLE_REQUEST_EXCEPTIONS as exc:
            last_error = exc
            reset_request_session()
        except requests.exceptions.HTTPError as exc:
            last_error = exc
            response = getattr(exc, 'response', None)
            status_code = getattr(response, 'status_code', None)
            if status_code not in RETRYABLE_STATUS_CODES:
                raise
            reset_request_session()

        if attempt < max_retries:
            wait_seconds = retry_wait_seconds(attempt)
            print(f'第 {page} 页请求失败，{wait_seconds:.1f} 秒后重试 {attempt}/{max_retries}：{last_error!r}')
            time.sleep(wait_seconds)

    raise PageRequestError(f'第 {page} 页连续 {max_retries} 次请求失败：{last_error!r}') from last_error


## 单元格 7：浏览器备用说明

当前版本不再使用 Playwright 抓 HTML。最后保留一个空的关闭函数，只是为了兼容原 notebook 的运行顺序。


In [66]:
# 单元格 7：浏览器备用说明
# 说明：当前版本不再依赖 Playwright 解析页面 HTML；保留关闭函数是为了兼容最后一个单元格。


def close_playwright_browser():
    """当前接口版不启动 Playwright，这里保留为空操作。"""
    return None


## 单元格 8：解析接口返回数据

职位列表来自接口返回的 `data.list`，总数来自 `data.count`，是否最后一页来自 `data.isEndPage`。


In [67]:
# 单元格 8：解析接口返回数据
# 说明：职位列表在接口返回的 data.list 中，count 是接口给出的结果数量，isEndPage 表示是否最后一页。


def extract_position_page_data(data: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], int, bool]:
    """从接口 data 中提取职位列表、结果数量和是否最后一页。"""
    jobs = data.get('list') or []
    count = int(data.get('count') or 0)
    is_end_page = bool(data.get('isEndPage'))
    return jobs, count, is_end_page


## 单元格 9：基础清洗工具

后面整理字段时，会遇到列表、字典、空值、重复标签等情况。本单元格放通用小函数，避免后面的字段整理代码太乱。


In [68]:
# 单元格 9：基础清洗工具
# 说明：这些函数只做文本清洗、路径取值、列表合并，不涉及具体业务字段。


def get_path(obj: Any, path: List[str], default: Any = '') -> Any:
    """从多层字典里安全取值，路径不存在就返回 default。"""
    cur = obj
    for key in path:
        if isinstance(cur, dict) and key in cur:
            cur = cur[key]
        else:
            return default
    return default if cur is None else cur


def clean_text(value: Any) -> str:
    """把任意值转成去掉多余空白的字符串。"""
    if value is None:
        return ''
    if isinstance(value, str):
        return ' '.join(value.split())
    return str(value)


def unique_join(values: List[Any], sep: str = ' | ') -> str:
    """去空、去重，并用分隔符拼接。"""
    seen = set()
    result = []
    for value in values:
        text = clean_text(value)
        if not text or text in seen:
            continue
        seen.add(text)
        result.append(text)
    return sep.join(result)


def parse_json_field(value: Any) -> Dict[str, Any]:
    """解析某些字段里嵌套的 JSON 字符串，例如 cardCustomJson。"""
    if not value or not isinstance(value, str):
        return {}
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return {}


## 单元格 10：标签提取工具

岗位标签来源很多：`showSkillTags`、`jobSkillTags`、福利、职位关键词、职位详情里的标签等。本单元格把这些不同格式统一成文本列表。


In [69]:
# 单元格 10：标签提取工具
# 说明：尽可能把岗位相关标签都保留下来，后面你可以自己筛选需要的列。


def list_values(
    items: Any,
    keys: Tuple[str, ...] = ('name', 'value', 'tag', 'itemValue', 'title', 'text', 'label', 'description'),
) -> List[str]:
    """从列表/字典/普通值中提取标签文本。"""
    if not items:
        return []
    if isinstance(items, dict):
        items = [items]
    if isinstance(items, (str, int, float)):
        return [clean_text(items)]

    values = []
    for item in items:
        if isinstance(item, dict):
            for key in keys:
                if item.get(key) not in (None, ''):
                    values.append(item[key])
                    break
        else:
            values.append(item)
    return [clean_text(v) for v in values if clean_text(v)]


def key_value_items(items: Any, name_key: str = 'name', value_key: str = 'value') -> List[str]:
    """把 [{'name': '职位底薪', 'value': '8000元/月'}] 转成 '职位底薪:8000元/月'。"""
    if not items:
        return []

    values = []
    for item in items:
        if not isinstance(item, dict):
            values.append(clean_text(item))
            continue

        name = clean_text(item.get(name_key, ''))
        value = clean_text(item.get(value_key, ''))
        if name and value and name != value:
            values.append(f'{name}:{value}')
        elif name:
            values.append(name)
        elif value:
            values.append(value)
    return values


def collect_all_tags(job: Dict[str, Any]) -> str:
    """汇总一个岗位里能找到的所有标签。"""
    jd = job.get('jobDetailData') or {}
    custom = jd.get('customAttributeInfo') or {}
    desc = get_path(jd, ['position', 'desc'], {})

    tags = []
    tags += list_values(job.get('jobSkillTags'), ('name', 'value', 'tag'))
    tags += list_values(job.get('skillLabel'), ('value', 'name', 'tag'))
    tags += list_values(job.get('showSkillTags'), ('tag', 'name', 'value'))
    tags += list_values(get_path(job, ['jobKeyword', 'keywords'], []), ('itemValue', 'name', 'tag', 'value'))
    tags += list_values(desc.get('labels'))
    tags += list_values(desc.get('welfareLabel'))
    tags += list_values(desc.get('welfareTags'))
    tags += list_values(job.get('welfareLabel'))
    tags += list_values(custom.get('reportItems'))
    tags += key_value_items(custom.get('welfareItems'))
    tags += key_value_items(custom.get('workTimeItems'))
    tags += list_values(job.get('searchTagList'))
    tags += list_values(job.get('commercialLabel'))
    tags += list_values(job.get('orgCommercialTags'))
    tags += list_values(job.get('companyScaleTypeTagsNew'))

    if job.get('tagABC'):
        tags.append(job.get('tagABC'))
    return unique_join(tags)


## 单元格 11：把单个岗位整理成一行

原始 JSON 很深，不方便直接看。本单元格把一个岗位字典整理成表格中的一行，同时保留 `原始JSON`，避免后续筛字段时丢信息。


In [70]:
# 单元格 11：单岗位字段整理
# 说明：这里是字段取舍最集中的地方。想新增列，可以照着 return 字典继续加。


def flatten_job(job: Dict[str, Any], page: int, keyword: str, city: str, city_code: str) -> Dict[str, Any]:
    """把一个职位 JSON 整理成一行表格数据。"""
    jd = job.get('jobDetailData') or {}
    pos = jd.get('position') or {}
    base = pos.get('base') or {}
    desc = pos.get('desc') or {}
    custom = jd.get('customAttributeInfo') or {}
    workloc = jd.get('workLocation') or {}
    staff_detail = jd.get('staff') or {}
    staff_card = job.get('staffCard') or {}
    state = get_path(jd, ['stateInfo', 'state'], {})
    verify_basic = get_path(jd, ['verifyTheTruth', 'basic'], {})
    card = parse_json_field(job.get('cardCustomJson'))

    skill_tags = []
    skill_tags += list_values(job.get('jobSkillTags'), ('name',))
    skill_tags += list_values(job.get('skillLabel'), ('value', 'name'))
    skill_tags += list_values(job.get('showSkillTags'), ('tag', 'name', 'value'))

    welfare_tags = []
    welfare_tags += list_values(desc.get('welfareLabel'))
    welfare_tags += list_values(desc.get('welfareTags'))
    welfare_tags += list_values(job.get('welfareLabel'))
    welfare_tags += key_value_items(custom.get('welfareItems'))

    return {
        '搜索关键词': keyword,
        '搜索城市': city,
        '城市代码': city_code,
        '页码': page,
        '职位ID': job.get('jobId') or base.get('positionId'),
        '职位编号': job.get('number') or base.get('positionNumber'),
        '职位名称': job.get('name') or base.get('positionName'),
        '职位URL': job.get('positionURL') or job.get('positionUrl') or base.get('positionUrl'),
        '薪资': job.get('salary60') or base.get('salary') or card.get('salary60'),
        '薪资原始区间': job.get('salaryReal') or base.get('salaryReal'),
        '薪资类型': job.get('salaryType', ''),
        '薪资发放次数': job.get('salaryCount', ''),
        '工作城市': job.get('workCity') or workloc.get('positionWorkCity'),
        '行政区': job.get('cityDistrict') or workloc.get('positionCityDistrict'),
        '商圈/街道': unique_join([job.get('tradingArea'), job.get('streetName'), workloc.get('tradingArea'), workloc.get('streetName')]),
        '工作地点展示': workloc.get('address') or card.get('address') or job.get('workCity'),
        '详细地址': workloc.get('workAddress', ''),
        '经度': workloc.get('longitude', ''),
        '纬度': workloc.get('latitude', ''),
        '经验要求': job.get('workingExp') or base.get('positionWorkingExp'),
        '学历要求': job.get('education') or base.get('education'),
        '工作类型': job.get('workType') or base.get('workType'),
        '工作模式': job.get('workMode') or state.get('workModeDesc') or state.get('workMode'),
        '职位类别': unique_join([
            job.get('jobTypeLevelName'),
            job.get('subJobTypeLevelName'),
            get_path(jd, ['jobType', 'jobTypeLevelName']),
            get_path(jd, ['jobType', 'subJobTypeLevelName']),
        ]),
        '公司名称': job.get('companyName') or card.get('companyName'),
        '公司编号': job.get('companyNumber'),
        '公司URL': job.get('companyUrl'),
        '公司Logo': job.get('companyLogo'),
        '公司规模': job.get('companySize'),
        '公司性质': job.get('propertyName') or job.get('property'),
        '融资阶段': job.get('financingStage') or card.get('strengthLabel'),
        '行业': job.get('industryName'),
        '发布时间': job.get('publishTime') or get_path(pos, ['date', 'positionPublishTime']) or get_path(pos, ['date', 'positionUpdateTimeText']),
        '首次发布时间': job.get('firstPublishTime') or get_path(pos, ['date', 'firstPublishTime']),
        '发布日期文本': get_path(pos, ['date', 'positionUpdateTimeText']),
        '是否新职位': job.get('isNewPosition'),
        '招聘人数': job.get('recruitNumber') or base.get('recruitNumber'),
        'HR姓名': staff_card.get('staffName') or staff_detail.get('staffName'),
        'HR职位': staff_card.get('hrJob') or staff_detail.get('hrJob'),
        'HR状态': staff_card.get('hrStateInfo') or staff_detail.get('hrStateInfo'),
        'HR活跃标签': unique_join(staff_detail.get('activityLevel') or []),
        '职位标签汇总': collect_all_tags(job),
        '技能标签': unique_join(skill_tags),
        '搜索命中关键词': unique_join(list_values(get_path(job, ['jobKeyword', 'keywords'], []), ('itemValue', 'name', 'value'))),
        '福利标签': unique_join(welfare_tags),
        '福利明细': unique_join(key_value_items(custom.get('welfareItems'))),
        '工作时间': unique_join(key_value_items(custom.get('workTimeItems'))),
        '报告项/保障项': unique_join(list_values(custom.get('reportItems'))),
        '职位描述': desc.get('description') or get_path(pos, ['desc', 'description']),
        '职位亮点': desc.get('descriptionHighlight') or job.get('positionHighlight'),
        '职位摘要': job.get('jobSummary', ''),
        '认证/守护信息': verify_basic.get('description') or get_path(jd, ['secure', 'safeCenter', 'title']),
        '原始JSON': json.dumps(job, ensure_ascii=False),
    }


## 单元格 12：主爬取函数

这一格负责翻页、请求 JSON 接口、解析职位列表，并返回三个 DataFrame：`clean_df` 是最终导出的整理字段；`raw_df` 和 `meta_df` 只用于检查原始字段和每页抓取情况，不会在导出单元格里保存成文件。


In [71]:
# 单元格 12：主爬取函数
# 返回值说明：clean_df 是最终导出的整理表；raw_df 和 meta_df 只用于检查字段和页码抓取情况，不再导出文件。


def crawl_zhaopin(
    keyword: str,
    city: str,
    city_code: Optional[str] = None,
    start_page: int = 1,
    max_pages: int = 1,
    page_size: int = 20,
    extra_params: Optional[Dict[str, Any]] = None,
    empty_page_stop: int = 2,
):
    """按照关键词、城市和页码抓取智联招聘岗位。"""
    resolved_city_code = resolve_city_code(city, city_code)
    all_jobs = []
    meta_rows = []
    stop_page = start_page + max_pages - 1
    empty_page_count = 0

    for page_no in tqdm(range(start_page, stop_page + 1), desc='抓取页码'):
        page_url = build_search_page_url(keyword, resolved_city_code, page_no, extra_params=extra_params)
        print(f'抓取第 {page_no} 页：{page_url}')

        try:
            data, payload, api_url = fetch_position_page(
                keyword=keyword,
                city_code=resolved_city_code,
                page=page_no,
                page_size=page_size,
                extra_params=extra_params,
                timeout=REQUEST_TIMEOUT,
            )
        except PageRequestError as exc:
            failed_payload = build_search_payload(keyword, resolved_city_code, page_no, page_size, extra_params)
            meta_rows.append({
                'page': page_no,
                'url': page_url,
                'api_url': SEARCH_API_URL,
                'jobs': 0,
                'reported_pages': '',
                'position_count': '',
                'is_end_page': '',
                'is_verification': '',
                'task_id': '',
                'error': repr(exc),
                'request_payload': json.dumps(failed_payload, ensure_ascii=False),
            })
            print(f'第 {page_no} 页多次重试仍失败，保留当前城市已抓到的数据并停止该城市：{exc!r}')
            break

        jobs, position_count, is_end_page = extract_position_page_data(data)
        reported_pages = (position_count + page_size - 1) // page_size if position_count else 0

        meta_rows.append({
            'page': page_no,
            'url': page_url,
            'api_url': api_url,
            'jobs': len(jobs),
            'reported_pages': reported_pages,
            'position_count': position_count,
            'is_end_page': int(is_end_page),
            'is_verification': data.get('isVerification', ''),
            'task_id': data.get('taskId', ''),
            'error': '',
            'request_payload': json.dumps(payload, ensure_ascii=False),
        })

        if not jobs:
            empty_page_count += 1
            print(f'当前页没有职位，连续空页数：{empty_page_count}/{empty_page_stop}')
            if empty_page_count >= empty_page_stop:
                print('连续空页达到停止阈值，停止抓取。')
                break
            if page_no < stop_page:
                time.sleep(random.uniform(*PAGE_SLEEP_RANGE))
            continue

        empty_page_count = 0
        for job in jobs:
            item = dict(job)
            item['_page'] = page_no
            all_jobs.append(item)

        if is_end_page:
            print('接口提示已经到达最后一页。')
            break

        if page_no < stop_page:
            time.sleep(random.uniform(*PAGE_SLEEP_RANGE))

    clean_rows = [
        flatten_job(job, page=job.get('_page', ''), keyword=keyword, city=city, city_code=resolved_city_code)
        for job in all_jobs
    ]
    clean_df = pd.DataFrame(clean_rows)
    raw_df = pd.json_normalize(all_jobs, sep='.') if all_jobs else pd.DataFrame()
    meta_df = pd.DataFrame(meta_rows)
    return clean_df, raw_df, meta_df


## 单元格 13：执行抓取并自动导出

运行这一格才会真正访问接口。当前配置会按 `CITY_LIST` 批量抓取网页上方城市入口中的每个城市，每个城市抓取 `START_PAGE` 到 `START_PAGE + MAX_PAGES - 1` 页，也就是默认每个城市 1-5 页。

每抓完一个城市，就会立即导出该城市 CSV，结构为 `项目根目录/data/关键词/zhaopin_城市_关键词.csv`。这样即使中途某个城市失败，前面已经抓完的城市文件也会保留下来。


In [72]:
# 单元格 13：执行抓取并自动导出
# 说明：每抓完一个城市就立即保存一个 CSV；如果中途失败，已保存的城市文件不会丢失。

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_name(text: str) -> str:
    """把文件名里不适合保存的字符替换成下划线。"""
    return re.sub(r'[\/:*?"<>|\s]+', '_', str(text)).strip('_')


# 使用固定文件名；重复运行时会覆盖同名城市 CSV。
keyword_dir = OUTPUT_DIR / safe_name(KEYWORD)
keyword_dir.mkdir(parents=True, exist_ok=True)

city_clean_frames = []
city_summary_rows = []
exported_files = []
failed_cities = []

for city_name in CITY_LIST:
    print(f'\n===== 开始抓取城市：{city_name}，页码 {START_PAGE}-{START_PAGE + MAX_PAGES - 1} =====')
    try:
        current_clean_df, _, current_meta_df = crawl_zhaopin(
            keyword=KEYWORD,
            city=city_name,
            city_code=None,
            start_page=START_PAGE,
            max_pages=MAX_PAGES,
            page_size=PAGE_SIZE,
            extra_params=EXTRA_PARAMS,
            empty_page_stop=EMPTY_PAGE_STOP,
        )
    except Exception as exc:
        failed_cities.append({'城市': city_name, '错误': repr(exc)})
        print(f'城市 {city_name} 抓取失败：{exc!r}')
        continue

    page_count = int(current_meta_df['page'].nunique()) if not current_meta_df.empty and 'page' in current_meta_df.columns else 0
    row_count = len(current_clean_df)
    error_messages = []
    if not current_meta_df.empty and 'error' in current_meta_df.columns:
        error_messages = [msg for msg in current_meta_df['error'].dropna().astype(str).tolist() if msg]
    status = '成功'
    if error_messages:
        status = '部分成功' if row_count else '失败'

    city_summary_rows.append({
        '城市': city_name,
        '抓取页数': page_count,
        '记录数': row_count,
        '状态': status,
        '错误': error_messages[-1] if error_messages else '',
    })

    if current_clean_df.empty:
        print(f'城市 {city_name} 没有抓到数据，跳过导出。')
        continue

    output_csv = keyword_dir / f'{safe_name(city_name)}_{safe_name(KEYWORD)}.csv'
    current_clean_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    exported_files.append(output_csv)
    city_clean_frames.append(current_clean_df)
    print(f'城市 {city_name} 已导出：{output_csv.resolve()}')

clean_df = pd.concat(city_clean_frames, ignore_index=True) if city_clean_frames else pd.DataFrame()
summary_df = pd.DataFrame(city_summary_rows)
failed_df = pd.DataFrame(failed_cities)

print('\n===== 抓取完成 =====')
print(f'计划城市数：{len(CITY_LIST)}')
print(f'成功导出城市数：{len(exported_files)}')
print(f'整理后总记录数：{len(clean_df)}')

if not summary_df.empty:
    display(summary_df)
if not failed_df.empty:
    print('以下城市抓取失败：')
    display(failed_df)



===== 开始抓取城市：全国，页码 1-50 =====


抓取页码:   0%|          | 0/50 [00:00<?, ?it/s]

抓取第 1 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=1&jl=489&order=4


抓取页码:   2%|▏         | 1/50 [00:03<03:04,  3.76s/it]

抓取第 2 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=2&jl=489&order=4


...

抓取第 50 页：https://www.zhaopin.com/sou/?kw=%E6%95%B0%E6%8D%AE%E5%BC%80%E5%8F%91&p=50&jl=489&order=4


抓取页码:  98%|█████████▊| 49/50 [02:16<00:02,  2.79s/it]

接口提示已经到达最后一页。


城市 全国 已导出：D:\桌面\实训项目\data\数据开发\全国_数据开发.csv

===== 抓取完成 =====
计划城市数：1
成功导出城市数：1
整理后总记录数：1000


,城市,抓取页数,记录数,状态,错误
0,全国,50,1000,成功,
